<a href="https://colab.research.google.com/github/Pranayshukla0610/ML-projects-portfolio/blob/main/Tesla_Financial_Intelligence_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import os

from math import pi

from bokeh.io import output_notebook, show
from bokeh.io import curdoc



from bokeh.layouts import (
    row,
    column
)

from bokeh.models import (
    ColumnDataSource,
    HoverTool,
    Div,
    Select,
    DateRangeSlider
)

In [2]:
output_notebook()

In [3]:
BACKGROUND = "#0F172A"
CARD = "#1E293B"
TEXT = '#F8FAFC'
GRID = '#334155'

WIDTH = 650
HEIGHT = 400

In [4]:
companies = {
    'Tesla':'TSLA',
    'Netflix':'NFLX',
    'NVIDIA':'NVDA',
    'Meta':'META',
    'Amazon':'AMZN'
}

In [5]:
ticker = companies['Tesla']

In [6]:
df = yf.download(
    ticker,
    period = '2y',
    interval = '1d'
)

/tmp/ipykernel_6083/3152141179.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  1 of 1 completed


In [7]:
df.reset_index(inplace = True)

In [8]:
df.columns = df.columns.get_level_values(0)

In [9]:
df['Returns'] = (
    df['Close'].pct_change()
)

In [10]:
df['MA20'] = (
    df['Close'].rolling(20).mean()
)

df['MA50'] = (
    df['Close'].rolling(50).mean()
)

df['MA100'] = (
    df['Close'].rolling(100).mean()
)

In [11]:
df['Volatility'] = (
    df['Returns'].rolling(20).std()
)

In [12]:
#BOLLINGER BANDS

df['Upper'] = (
    df['MA20']
    + 2 * df['Volatility']
)

df['Lower'] = (
    df['MA20']
    - 2 * df['Volatility']
)

In [13]:
#RELATIVE STRENGTH INDEX (RSI)

delta = df['Close'].diff()

gain = delta.where(
    delta > 0,
    0
)

loss = -delta.where(
    delta < 0,
    0
)

avg_gain = gain.rolling(14).mean()

avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df['RSI'] = (
    100 - (100 / (1 + rs))
)

###RSI Value	Meaning
###>70	      Overbought
###<30	      Oversold

In [14]:
investment = 10000

shares = (
    investment
    /
    df['Close'].iloc[0]
)

df['Portfolio'] = (
    shares
    * df['Close']
)

In [15]:
df.dropna(inplace=True)

In [16]:
df.head()

Price,Date,Close,High,Low,Open,Volume,Returns,MA20,MA50,MA100,Volatility,Upper,Lower,RSI,Portfolio
99,2024-10-01,258.019989,263.980011,248.529999,262.670013,87397600,-0.013798,237.530998,222.973401,211.3593,0.033941,237.598880,237.463117,73.946481,15003.778984
100,2024-10-02,249.020004,251.160004,241.500000,247.550003,93983900,-0.034881,239.011498,223.026201,212.1298,0.034573,239.080644,238.942353,63.774571,14480.432779
101,2024-10-03,240.660004,249.789993,237.809998,244.479996,80729200,-0.033572,239.535999,223.519601,212.8517,0.034209,239.604416,239.467581,56.680849,13994.301445
102,2024-10-04,250.080002,250.960007,244.580002,246.690002,86573200,0.039142,241.503499,224.116201,213.6336,0.028265,241.560030,241.446968,63.948761,14542.071295
103,2024-10-07,240.830002,249.830002,240.699997,249.000000,68113300,-0.036988,242.731499,224.536801,214.2664,0.029728,242.790955,242.672042,57.068068,14004.186784


In [17]:
source = ColumnDataSource()

In [18]:
latest_price = round(
    df['Close'].iloc[-1],
    2
)

daily_return = round(
    df['Returns'].iloc[-1] * 100,
    2
)

volatility = round(
    df['Volatility'].iloc[-1] * 100,
    2
)

portfolio_value = round(
    df['Portfolio'].iloc[-1],
    2
)

profit = round(
    portfolio_value - investment,
    2
)

In [19]:
risk_free_ratio = 0.02

sharpe_ratio = round(
    (
        df['Returns'].mean() * 252
        - risk_free_ratio
    )
    /
    (
        df['Returns'].std()
        * np.sqrt(252)
    ),
    2
)

In [20]:
def create_kpi(title, value, color):

    return Div(text=f"""

    <div style="
    background:{color};
    padding:20px;
    border-radius:18px;
    text-align:center;
    width:220px;
    height:120px;
    box-shadow:0px 4px 15px rgba(0,0,0,0.5);
    ">

    <h2 style="
    color:white;
    font-family:Arial;
    ">
    {title}
    </h2>

    <h1 style="
    color:white;
    ">
    {value}
    </h1>

    </div>

    """)

In [21]:
kpi1 = create_kpi(
    "Live Price",
    f"${latest_price}",
    "#2563EB"
)

kpi2 = create_kpi(
    "Daily Return",
    f"{daily_return}%",
    "#059669"
)

kpi3 = create_kpi(
    "Volatility",
    f"{volatility}%",
    "#DC2626"
)

kpi4 = create_kpi(
    "Portfolio Value",
    f"${portfolio_value}",
    "#7C3AED"
)

kpi5 = create_kpi(
    "Profit/Loss",
    f"${profit}",
    "#EA580C"
)

kpi6 = create_kpi(
    "Sharpe Ratio",
    f"{sharpe_ratio}",
    "#0891B2"
)

In [22]:
company_selector = Select(
    title = "Select Company",
    value = "Tesla",
    options = list(companies.keys())
)

In [23]:
date_slider = DateRangeSlider(
    title = "Select Date Range",
    start = df['Date'].min(),
    end = df['Date'].max(),
    value=(
        df['Date'].min(),
        df['Date'].max()
    ),
    width = 500
)

In [24]:
inc = df['Close'] > df['Open']
dec = df['Open'] > df['Close']

In [25]:
w = 12 * 60 * 60 * 1000

In [26]:
from bokeh.plotting import figure

p1 = figure(
    x_axis_type = 'datetime',
    width = WIDTH,
    height = HEIGHT,
    title = 'Tesla Candlestick Analysis',
    background_fill_color = CARD,
    border_fill_color = BACKGROUND,
    toolbar_location = 'above'
)

In [27]:
p1.segment(
    df['Date'],
    df['High'],
    df['Date'],
    df['Low'],
    color="white"
)

GlyphRenderer(id='p1074', ...)

In [28]:
p1.vbar(
    x=df.loc[inc,'Date'],
    width=w,
    top=df.loc[inc,'Close'],
    bottom=df.loc[inc,'Open'],
    fill_color="#10B981",
    line_color="#10B981"
)

GlyphRenderer(id='p1085', ...)

In [29]:
p1.vbar(
    x=df.loc[dec,'Date'],
    width=w,
    top=df.loc[dec,'Open'],
    bottom=df.loc[dec,'Close'],
    fill_color="#EF4444",
    line_color="#EF4444"
)

GlyphRenderer(id='p1096', ...)

In [30]:
p2 = figure(
    x_axis_type="datetime",
    width=WIDTH,
    height=HEIGHT,
    title="Moving Average Analytics",
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [31]:
p2.line(
    df['Date'],
    df['Close'],
    line_width=2,
    color="white",
    legend_label="Close Price"
)

GlyphRenderer(id='p1156', ...)

In [32]:
p2.line(
    df['Date'],
    df['MA20'],
    line_width=3,
    color="#3B82F6",
    legend_label="MA20"
)

GlyphRenderer(id='p1169', ...)

In [33]:
p2.line(
    df['Date'],
    df['MA50'],
    line_width=3,
    color="#F59E0B",
    legend_label="MA50"
)

GlyphRenderer(id='p1181', ...)

In [34]:
p2.line(
    df['Date'],
    df['MA100'],
    line_width=3,
    color="#8B5CF6",
    legend_label="MA100"
)

GlyphRenderer(id='p1193', ...)

In [35]:
p3 = figure(
    x_axis_type="datetime",
    width=WIDTH,
    height=HEIGHT,
    title="Trading Volume Analysis",
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [36]:
p3.vbar(
    x=df['Date'],
    top=df['Volume'],
    width=w,
    color="#06B6D4"
)

GlyphRenderer(id='p1254', ...)

In [37]:
p4 = figure(
    x_axis_type="datetime",
    width=WIDTH,
    height=HEIGHT,
    title="Volatility Risk Analytics",
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [38]:
p4.line(
    df['Date'],
    df['Volatility'],
    line_width=3,
    color="#EF4444"
)

GlyphRenderer(id='p1314', ...)

In [39]:
p5 = figure(
    x_axis_type="datetime",
    width=WIDTH,
    height=HEIGHT,
    title="RSI Momentum Analysis",
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [40]:
p5.line(
    df['Date'],
    df['RSI'],
    line_width=3,
    color="#F97316"
)

GlyphRenderer(id='p1374', ...)

In [41]:
p5.line(
    df['Date'],
    [70]*len(df),
    color="red",
    line_dash="dashed"
)

p5.line(
    df['Date'],
    [30]*len(df),
    color="green",
    line_dash="dashed"
)

GlyphRenderer(id='p1394', ...)

In [42]:
p6 = figure(
    x_axis_type="datetime",
    width=WIDTH,
    height=HEIGHT,
    title="Portfolio Growth Analytics",
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [43]:
p6.line(
    df['Date'],
    df['Portfolio'],
    line_width=4,
    color="#22C55E"
)

GlyphRenderer(id='p1454', ...)

In [44]:
hist, edges = np.histogram(
    df['Returns'].dropna(),
    bins=40
)

In [45]:
p7 = figure(
    width=WIDTH,
    height=HEIGHT,
    title="Return Distribution",
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [46]:
p7.quad(
    top=hist,
    bottom=0,
    left=edges[:-1],
    right=edges[1:],
    fill_color="#8B5CF6",
    line_color="white"
)

GlyphRenderer(id='p1500', ...)

In [47]:
hover = HoverTool(
    tooltips=[

        ("Date", "@Date{%F}"),
        ("Open", "@Open"),
        ("Close", "@Close"),
        ("High", "@High"),
        ("Low", "@Low"),
        ("Volume", "@Volume")

    ],

    formatters={
        '@Date':'datetime'
    }
)

In [48]:
p1.add_tools(hover)

In [49]:
plots = [p1,p2,p3,p4,p5,p6,p7]

for p in plots:

    p.title.text_color = TEXT

    p.xaxis.major_label_text_color = TEXT
    p.yaxis.major_label_text_color = TEXT

    p.xaxis.axis_label_text_color = TEXT
    p.yaxis.axis_label_text_color = TEXT

    p.grid.grid_line_color = GRID

In [50]:
title = Div(text="""

<div style="

background:linear-gradient(
90deg,
#111827,
#2563EB
);

padding:25px;

border-radius:18px;

box-shadow:0px 4px 15px rgba(0,0,0,0.5);

">

<h1 style="

color:white;
text-align:center;
font-size:40px;
font-family:Arial;

">

TESLA FINANCIAL INTELLIGENCE DASHBOARD

</h1>

<p style="
color:white;
text-align:center;
font-size:18px;
">

Real-Time Stock Analytics | Risk Analytics |
Portfolio Monitoring | Quantitative Finance

</p>

</div>

""")

In [51]:
subtitle = Div(text="""

<h2 style="
color:white;
padding:10px;
">

Interactive Financial Analytics Platform
using Python + Bokeh

</h2>

""")

In [52]:
market_status = Div(text="""

<div style="

background:#059669;
padding:20px;
border-radius:15px;
width:250px;
text-align:center;

">

<h2 style="color:white;">

Market Status

</h2>

<h1 style="color:white;">

LIVE

</h1>

</div>

""")


In [53]:
risk_card = Div(text=f"""

<div style="

background:#DC2626;
padding:20px;
border-radius:15px;
width:250px;
text-align:center;

">

<h2 style="color:white;">

Risk Level

</h2>

<h1 style="color:white;">

{round(volatility,2)}%

</h1>

</div>

""")

In [54]:
return_card = Div(text=f"""

<div style="

background:#7C3AED;
padding:20px;
border-radius:15px;
width:250px;
text-align:center;

">

<h2 style="color:white;">

Expected Return

</h2>

<h1 style="color:white;">

{daily_return}%

</h1>

</div>

""")

In [55]:
divider = Div(text="""

<hr style="
border:2px solid #334155;
">

""")

In [56]:
latest = yf.download(
    ticker,
    period="1d",
    interval="1m"
    )

/tmp/ipykernel_6083/1766708287.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  latest = yf.download(
[*********************100%***********************]  1 of 1 completed


In [57]:
latest_price = latest['Close'].iloc[-1]

In [58]:
from bokeh.io import output_notebook, show

output_notebook()

dashboard = column(

    title,

    subtitle,

    divider,

    row(kpi1, kpi2, kpi3),

    row(kpi4, kpi5, kpi6),

    row(
        market_status,
        risk_card,
        return_card
    ),

    divider,

    row(
        company_selector,
        date_slider
    ),

    row(p1, p2),

    row(p3, p4),

    row(p5, p6),

    p7

)

show(dashboard)

ERROR:bokeh.core.validation.check:E-1027 (REPEATED_LAYOUT_CHILD): The same model can't be used multiple times in a layout: Column(id='p1519', ...)
